In [ ]:
!pip install transformers seqeval evaluate accelerate -U
!pip install transformers seqeval evaluate accelerate pytorch-crf -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 52.1 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=7adda0f0f91421c2f7d54591a2a61e6f13a15d59f0e300874e0d7e87cfa59394
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.15.1
    Uninstalling transformers-5.15.1:
      Successfully uninstalled transformers-5.15.1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
label_path = '/content/drive/MyDrive/datasetViMedNER/traindata/labels.txt'
train_path = '/content/drive/MyDrive/datasetViMedNER/traindata/train.txt'
dev_path = '/content/drive/MyDrive/datasetViMedNER/traindata/dev.txt'

unique_tags = []
with open(label_path, "r", encoding = "utf-8") as f :
    for line in f:
        line.strip()
        if line.strip():
          unique_tags.append(line.strip())

label2id = {tag: idx for idx, tag in enumerate(unique_tags)}
id2label = {idx: tag for idx, tag in enumerate(unique_tags)}

def load_conll_data(file_path):
    sentences = []
    current_sentence = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    word, tag = parts[0], parts[1]
                    current_sentence.append((word, tag))
        if current_sentence:
            sentences.append(current_sentence)
    return sentences

train_sentences = load_conll_data(train_path)
dev_sentences = load_conll_data(dev_path)

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

class ViMedNERDataset(Dataset):
    def __init__(self, sentences, tokenizer, label2id, max_length=128):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence_data = self.sentences[idx]
        words = [item[0] for item in sentence_data]
        tags = [item[1] for item in sentence_data]

        # Tokenize từng từ và theo dõi số lượng sub-token của mỗi từ
        tokenized_outputs = []
        label_ids = []

        # Thêm token [CLS] đầu câu
        tokenized_outputs.append(self.tokenizer.cls_token_id)
        label_ids.append(-100)

        for word, tag in zip(words, tags):
            # Tokenize từng từ lẻ (không dùng is_split_into_words để tránh lỗi word_ids)
            sub_tokens = self.tokenizer.tokenize(word)
            sub_token_ids = self.tokenizer.convert_tokens_to_ids(sub_tokens)

            if len(sub_token_ids) > 0:
                # Sub-token đầu tiên nhận nhãn thật
                tokenized_outputs.append(sub_token_ids[0])
                label_ids.append(self.label2id[tag])

                # Các sub-token phía sau của cùng 1 từ nhận -100
                for sub_id in sub_token_ids[1:]:
                    tokenized_outputs.append(sub_id)
                    label_ids.append(-100)

        # Thêm token [SEP] cuối câu
        tokenized_outputs.append(self.tokenizer.sep_token_id)
        label_ids.append(-100)

        # Cắt ngắn (Truncation) nếu vượt quá max_length
        if len(tokenized_outputs) > self.max_length:
            tokenized_outputs = tokenized_outputs[:self.max_length]
            label_ids = label_ids[:self.max_length]

        # Tạo attention mask (1 cho token thật, 0 cho padding)
        attention_mask = [1] * len(tokenized_outputs)

        # Padding (Đệm) cho đủ max_length
        padding_length = self.max_length - len(tokenized_outputs)
        if padding_length > 0:
            tokenized_outputs = tokenized_outputs + [self.tokenizer.pad_token_id] * padding_length
            label_ids = label_ids + [-100] * padding_length
            attention_mask = attention_mask + [0] * padding_length

        item = {
            "input_ids": torch.tensor(tokenized_outputs, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(label_ids, dtype=torch.long)
        }
        return item

# 1. Khởi tạo Tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

In [ ]:
import pickle
train_dataset_filepath = '/content/drive/MyDrive/datasetViMedNER/PreprocessedData/train_dataset.pkl'
dev_dataset_filepath = '/content/drive/MyDrive/datasetViMedNER/PreprocessedData/dev_dataset.pkl'

def load_dataset(file_path):
    with open(file_path,'rb') as f :
        dataset = pickle.load(f)
    return dataset


train_dataset = load_dataset(train_dataset_filepath)
dev_dataset = load_dataset(dev_dataset_filepath)

add CRF layer

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
from transformers.modeling_outputs import TokenClassifierOutput
from torchcrf import CRF

class PhoBERT_CRF(nn.Module):
    def __init__(self, model_checkpoint, num_labels):
        super().__init__()
        self.phobert = AutoModel.from_pretrained(model_checkpoint, add_pooling_layer = False)
        self.classifier = nn.Linear(self.phobert.config.hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first = True)

    def forward(self, input_ids, attention_mask, labels = None, **kwargs):
        outputs = self.phobert(input_ids = input_ids, attention_mask = attention_mask) # đưa qua phobert lấy ngữ cảnh câu

        sequence_output = outputs.last_hidden_state

        emissions = self.classifier(sequence_output) # đưa ra kết quả thô của từng nhãn

        loss = None
        if labels is not None:
            crf_mask = attention_mask.bool() # chuyển nhãn không hợp lệ thành True False để xác định những vị trí cần tính crf

            # đổi quy ước của labels khi đưa qua crf
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0 #các labels padding, sub_token chuyển nhãn 0

            loss = -self.crf(emissions, safe_labels, mask = crf_mask, reduction = 'mean')

        # tạo một ma trận kết quả phụ để mô hình có thể sử dụng kết quả
        crf_mask_decode = attention_mask.bool()
        decoded_paths = self.crf.decode(emissions, mask = crf_mask_decode)

        fake_logits = torch.zeros_like(emissions)
        for i, path in enumerate(decoded_paths):
            for j, tag_id in enumerate(path):
                fake_logits[i, j, tag_id] = 1.0

        return TokenClassifierOutput(loss=loss, logits=fake_logits)

replace CE by Focal Loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from transformers.modeling_outputs import TokenClassifierOutput

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha = None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
    def forward(self, logits, targets, mask):
        logits = logits.view(-1, logits.size(-1))
        targets = targets.view(-1)
        mask = mask.view(-1)

        logits = logits[mask]
        targets = targets[mask]

        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

class PhoBERT_Focal(nn.Module):
    def __init__(self, model_checkpoint, num_labels):
        super().__init__()
        self.num_labels = num_labels

        self.phobert = AutoModel.from_pretrained(model_checkpoint, add_pooling_layer=False)
        self.classifier = nn.Linear(self.phobert.config.hidden_size, num_labels)

        # Chỉ dùng Focal Loss, không có CRF
        self.focal_loss = FocalLoss(gamma=2.0)

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state

        # Điểm số dự đoán thô
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            # Mask bỏ qua các token -100
            loss_mask = (labels != -100)

            # Tính Focal Loss trực tiếp trên logits thay vì Cross Entropy
            loss = self.focal_loss(logits, labels, loss_mask)

        return TokenClassifierOutput(loss=loss, logits=logits)

In [ ]:
import numpy as np
import evaluate
seqeval = evaluate.load("seqeval") # Khai báo seqeval ở đây
def compute_metrics(p):
    predictions, labels = p

    # Biến ma trận xác suất (logits) thành con số ID nhãn dự đoán cao nhất
    predictions = np.argmax(predictions, axis=2)

    # Lọc bỏ các nhãn -100 ra khỏi quá trình tính toán metric
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


def compute_eval_classify_metrics(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=2)

    # 1. Lọc nhãn -100 và giữ nguyên cấu trúc List of Lists (dành cho Seqeval)
    true_predictions_seq = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels_seq = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # 2. Trải phẳng thành mảng 1 chiều (dành cho Sklearn)
    true_predictions_flat = [tag for sent in true_predictions_seq for tag in sent]
    true_labels_flat = [tag for sent in true_labels_seq for tag in sent]

    metrics_dict = {}

    # =========================================================
    # BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ HOÀN CHỈNH (SEQEVAL)
    # =========================================================
    results_seq = seqeval.compute(predictions=true_predictions_seq, references=true_labels_seq)
    table_entity = []

    for key, value in results_seq.items():
        if isinstance(value, dict):
            table_entity.append({
                "Thực thể (Entity)": key,
                "Precision": f"{value['precision']:.4f}",
                "Recall": f"{value['recall']:.4f}",
                "F1-Score": f"{value['f1']:.4f}",
                "Number (Entities)": value['number']
            })
            metrics_dict[f"entity_{key}_f1"] = value["f1"]

    table_entity.append({
        "Thực thể (Entity)": "OVERALL",
        "Precision": f"{results_seq['overall_precision']:.4f}",
        "Recall": f"{results_seq['overall_recall']:.4f}",
        "F1-Score": f"{results_seq['overall_f1']:.4f}",
        "Number (Entities)": "-"
    })
    metrics_dict["overall_f1"] = results_seq['overall_f1']

    print("\n" + "="*75)
    print("📊 BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ (STRICT ENTITY LEVEL)")
    print("="*75)
    display(pd.DataFrame(table_entity))

    # =========================================================
    # BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN RỜI RẠC (SKLEARN)
    # =========================================================
    report_tag = sklearn_report(true_labels_flat, true_predictions_flat, output_dict=True, zero_division=0)
    table_tag = []

    for key, value in report_tag.items():
        if key in ["accuracy", "macro avg", "weighted avg"]:
            continue
        table_tag.append({
            "Nhãn (Tag)": key,
            "Precision": f"{value['precision']:.4f}",
            "Recall": f"{value['recall']:.4f}",
            "F1-Score": f"{value['f1-score']:.4f}",
            "Number (Tokens)": int(value['support'])
        })

    overall_tag = report_tag["weighted avg"]
    table_tag.append({
        "Nhãn (Tag)": "OVERALL (Weighted)",
        "Precision": f"{overall_tag['precision']:.4f}",
        "Recall": f"{overall_tag['recall']:.4f}",
        "F1-Score": f"{overall_tag['f1-score']:.4f}",
        "Number (Tokens)": int(overall_tag['support'])
    })

    print("\n" + "="*75)
    print("📊 BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN (TOKEN TAG LEVEL: B-, I-, O)")
    print("="*75)
    display(pd.DataFrame(table_tag))

    return metrics_dict

In [ ]:
import numpy as np
import pandas as pd
import evaluate
from sklearn.metrics import classification_report as sklearn_report
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
from IPython.display import display
from torch.optim import AdamW
from transformers import EarlyStoppingCallback

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
checkpoint_name = "vinai/phobert-base-v2"
num_labels = len(unique_tags)

model_crf = PhoBERT_CRF(model_checkpoint= checkpoint_name , num_labels = num_labels)

crf_params = list(model_crf.crf.parameters())
phobert_crf_params = list(model_crf.phobert.parameters()) + list(model_crf.classifier.parameters())

optimizer_grouped_parameters_crf = [
    {'params': phobert_crf_params, 'lr': 3e-5}, # LR nhỏ cho mạng pre-trained
    {'params': crf_params, 'lr': 5e-4}          # LR lớn cho lớp khởi tạo từ đầu
]
optimizer_crf = AdamW(optimizer_grouped_parameters_crf, weight_decay=0.01)
# cấu hình tham số
training_args_crf = TrainingArguments(
    output_dir="./phobert_crf_phase2",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=15,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    max_grad_norm=1.0,
    report_to="none"
)

trainer_crf = Trainer(
    model=model_crf,
    args=training_args_crf,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer_crf, None), # Ép Trainer dùng Optimizer mới
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# Bắt đầu train tiếp
print("🚀 Bắt đầu huấn luyện Phase 2...")
trainer_crf.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 Bắt đầu huấn luyện Phase 2...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,15.129827,0.574584,0.649396,0.609704,0.884717
2,21.057361,11.863971,0.643398,0.707651,0.673996,0.896948
3,21.057361,11.315688,0.678107,0.700134,0.688945,0.896254
4,9.423737,11.330524,0.633875,0.733423,0.680025,0.888936
5,9.423737,11.534796,0.645917,0.732617,0.686541,0.889257
6,6.007719,11.957176,0.681864,0.718658,0.699778,0.896361
7,4.211604,12.947955,0.681097,0.726443,0.703040,0.897091
8,4.211604,13.530421,0.675557,0.741208,0.706861,0.898800
9,2.854809,13.843337,0.672906,0.733423,0.701863,0.896539
10,2.854809,14.581703,0.698327,0.728322,0.713009,0.899761


/usr/local/lib/python3.13/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=3718, training_loss=6.420020741118482, metrics={'train_runtime': 2821.5483, 'train_samples_per_second': 24.311, 'train_steps_per_second': 1.52, 'total_flos': 0.0, 'train_loss': 6.420020741118482, 'epoch': 13.0})

In [ ]:
model_focal = PhoBERT_CRF(model_checkpoint= checkpoint_name , num_labels = num_labels)

# ==========================================
# 2. KHAI BÁO OPTIMIZER MỚI
# ==========================================
# Khởi tạo lại AdamW để reset Optimizer States (Momentum, Variance)
optimizer_focal = AdamW(model_focal.parameters(), lr=4e-5, weight_decay=0.01)

# ==========================================
# 3. CẤU HÌNH TRAINER CHO PHASE 2
# ==========================================
training_args_focal = TrainingArguments(
    output_dir="./phobert_focal_phase2",
    eval_strategy="epoch",
    learning_rate=4e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=15,              # Đặt thêm 10 epoch cho Phase 2 (nó sẽ dừng sớm nếu F1 không tăng)
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    max_grad_norm=1.0,
    report_to="none"
)

trainer_focal = Trainer(
    model=model_focal,
    args=training_args_focal,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer_focal, None), # Truyền Optimizer mới vào đây
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# ==========================================
# 4. BẮT ĐẦU HUẤN LUYỆN TIẾP
# ==========================================
print("🚀 Bắt đầu huấn luyện Phase 2 cho Focal Loss...")
trainer_focal.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 Bắt đầu huấn luyện Phase 2 cho Focal Loss...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,14.919735,0.578847,0.649396,0.612095,0.888580
2,19.221510,12.689677,0.609922,0.696376,0.650288,0.894011
3,19.221510,12.507616,0.651298,0.693960,0.671952,0.899227
4,9.492188,12.990466,0.605821,0.720805,0.658330,0.889203
5,9.492188,13.829196,0.632644,0.730470,0.678046,0.893049
6,5.998175,14.257974,0.628466,0.730201,0.675525,0.893103
7,3.932434,15.979595,0.660368,0.713020,0.685685,0.899939
8,3.932434,16.068096,0.639963,0.733423,0.683513,0.895114
9,2.528784,17.529238,0.655316,0.718121,0.685282,0.896610
10,2.528784,18.329237,0.656702,0.719463,0.686651,0.895293


TrainOutput(global_step=4290, training_loss=5.31676739957227, metrics={'train_runtime': 2967.049, 'train_samples_per_second': 23.119, 'train_steps_per_second': 1.446, 'total_flos': 0.0, 'train_loss': 5.31676739957227, 'epoch': 15.0})

In [ ]:
import os
import torch

base_dir = "/content/drive/MyDrive"
crf_save_path = os.path.join(base_dir, "vimedner_final_crf")
focal_save_path = os.path.join(base_dir, "vimedner_final_focal")

os.makedirs(crf_save_path, exist_ok=True)
os.makedirs(focal_save_path, exist_ok=True)

# ==========================================
# 1. LƯU MÔ HÌNH CRF CHUẨN XÁC
# ==========================================
print("Đang lưu trọng số mô hình CRF...")
tokenizer.save_pretrained(crf_save_path)
torch.save(model_crf.state_dict(), os.path.join(crf_save_path, "pytorch_model.bin"))
print(f"✅ Đã lưu CRF tại: {crf_save_path}")

# ==========================================
# 2. LƯU MÔ HÌNH FOCAL LOSS CHUẨN XÁC
# ==========================================
print("Đang lưu trọng số mô hình Focal Loss...")
tokenizer.save_pretrained(focal_save_path)
torch.save(model_focal.state_dict(), os.path.join(focal_save_path, "pytorch_model.bin"))
print(f"✅ Đã lưu Focal Loss tại: {focal_save_path}")

Đang lưu trọng số mô hình CRF...
✅ Đã lưu CRF tại: /content/drive/MyDrive/vimedner_final_crf
Đang lưu trọng số mô hình Focal Loss...
✅ Đã lưu Focal Loss tại: /content/drive/MyDrive/vimedner_final_focal


In [ ]:
import numpy as np

def export_error_analysis(pred_results, output_name, original_sentences, id2label):
    """
    Hàm xuất file kết quả dự đoán (Error Analysis)
    - pred_results: Kết quả trả về từ trainer.predict()
    - output_name: Tên file muốn lưu (ví dụ: 'crf_dev_analysis')
    - original_sentences: Dữ liệu câu gốc (ví dụ: dev_sentences)
    - id2label: Dictionary ánh xạ ID sang tên nhãn
    """
    output_file = f"/content/drive/MyDrive/datasetViMedNER/{output_name}.txt"

    # Trích xuất predictions và labels
    # Bắt lỗi an toàn nếu predictions đang ở dạng ma trận 3 chiều (Base/Focal)
    if isinstance(pred_results.predictions, np.ndarray) and pred_results.predictions.ndim == 3:
        predictions = np.argmax(pred_results.predictions, axis=2)
    else:
        predictions = pred_results.predictions # Dành cho trường hợp output đã là mảng 1D

    labels = pred_results.label_ids
    predicted_tags_per_sentence = []

    # Loại bỏ các token -100 và ép kiểu int
    for prediction, label in zip(predictions, labels):
        pred_sent = [id2label[int(p)] for p, l in zip(prediction, label) if l != -100]
        predicted_tags_per_sentence.append(pred_sent)

    print(f"\n🔍 Đang xuất file phân tích lỗi: {output_name}...")

    with open(output_file, "w", encoding="utf-8") as f:
        for idx, (original_sent, predicted_tags) in enumerate(zip(original_sentences, predicted_tags_per_sentence)):
            # original_sent là list các tuple (word, true_tag)
            if len(original_sent) == len(predicted_tags):
                for (word, true_tag), pred_tag in zip(original_sent, predicted_tags):
                    f.write(f"{word}\t{true_tag}\t{pred_tag}\n")
                f.write("\n")
            else:
                print(f"⚠️ Cảnh báo: Lệch số lượng từ ở câu {idx} (Gốc: {len(original_sent)} | Dự đoán: {len(predicted_tags)}), bỏ qua...")

    print(f"✅ Đã xuất file thành công tại: {output_file}")

In [ ]:
# Gọi predict và truyền trực tiếp dưới dạng tuple (predictions, label_ids) vào hàm compute_metrics cũ
pred_result_crf = trainer_crf.predict(dev_dataset)
pred_result_focal = trainer_focal.predict(dev_dataset)

# 1. Đánh giá mô hình CRF
print("📊 Đánh giá PhoBERT + CRF:")
compute_metrics((pred_result_crf.predictions, pred_result_crf.label_ids))
compute_eval_classify_metrics((pred_result_crf.predictions, pred_result_crf.label_ids))

# 2. Đánh giá mô hình Focal Loss
print("\n📊 Đánh giá PhoBERT + Focal Loss:")
compute_metrics((pred_result_focal.predictions, pred_result_focal.label_ids))
compute_eval_classify_metrics((pred_result_focal.predictions, pred_result_focal.label_ids))

📊 Đánh giá PhoBERT + CRF:

📊 BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ (STRICT ENTITY LEVEL)


,Thực thể (Entity),Precision,Recall,F1-Score,Number (Entities)
0,bien_phap_chan_doan,0.6294,0.6642,0.6463,271
1,bien_phap_dieu_tri,0.6090,0.6248,0.6168,653
2,nguyen_nhan_benh,0.3294,0.3205,0.3249,259
3,ten_benh,0.8019,0.8613,0.8305,1795
4,trieu_chung_benh,0.6622,0.6640,0.6631,747
5,OVERALL,0.6983,0.7283,0.7130,-



📊 BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN (TOKEN TAG LEVEL: B-, I-, O)


,Nhãn (Tag),Precision,Recall,F1-Score,Number (Tokens)
0,B-bien_phap_chan_doan,0.7212,0.7159,0.7185,271
1,B-bien_phap_dieu_tri,0.6986,0.6922,0.6954,653
2,B-nguyen_nhan_benh,0.4286,0.3822,0.4041,259
3,B-ten_benh,0.8432,0.8986,0.8700,1795
4,B-trieu_chung_benh,0.7378,0.7082,0.7227,747
5,I-bien_phap_chan_doan,0.7585,0.5234,0.6193,942
6,I-bien_phap_dieu_tri,0.6844,0.5381,0.6025,1745
7,I-nguyen_nhan_benh,0.4203,0.3659,0.3912,850
8,I-ten_benh,0.8692,0.9250,0.8962,4585
9,I-trieu_chung_benh,0.7001,0.6367,0.6669,1522



📊 Đánh giá PhoBERT + Focal Loss:

📊 BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ (STRICT ENTITY LEVEL)


,Thực thể (Entity),Precision,Recall,F1-Score,Number (Entities)
0,bien_phap_chan_doan,0.5524,0.6421,0.5939,271
1,bien_phap_dieu_tri,0.5816,0.6493,0.6136,653
2,nguyen_nhan_benh,0.2701,0.3243,0.2947,259
3,ten_benh,0.7901,0.8535,0.8206,1795
4,trieu_chung_benh,0.6271,0.6506,0.6386,747
5,OVERALL,0.6636,0.7248,0.6928,-



📊 BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN (TOKEN TAG LEVEL: B-, I-, O)


,Nhãn (Tag),Precision,Recall,F1-Score,Number (Tokens)
0,B-bien_phap_chan_doan,0.6678,0.7122,0.6893,271
1,B-bien_phap_dieu_tri,0.7014,0.7412,0.7208,653
2,B-nguyen_nhan_benh,0.3993,0.4208,0.4098,259
3,B-ten_benh,0.8446,0.8992,0.8710,1795
4,B-trieu_chung_benh,0.7257,0.7082,0.7168,747
5,I-bien_phap_chan_doan,0.6942,0.5977,0.6423,942
6,I-bien_phap_dieu_tri,0.6702,0.5742,0.6185,1745
7,I-nguyen_nhan_benh,0.3959,0.4047,0.4002,850
8,I-ten_benh,0.8722,0.9180,0.8945,4585
9,I-trieu_chung_benh,0.7036,0.6255,0.6623,1522


{'entity_bien_phap_chan_doan_f1': np.float64(0.5938566552901025),
 'entity_bien_phap_dieu_tri_f1': np.float64(0.6136034732272069),
 'entity_nguyen_nhan_benh_f1': np.float64(0.2947368421052632),
 'entity_ten_benh_f1': np.float64(0.8205677557579003),
 'entity_trieu_chung_benh_f1': np.float64(0.6386333771353482),
 'overall_f1': np.float64(0.6928406466512702)}

In [ ]:
# xuất ra dự đoán so sánh
export_error_analysis(pred_result_crf, "crf_dev_analysis", dev_sentences, id2label)
export_error_analysis(pred_result_focal, "focal_dev_analysis", dev_sentences, id2label)


🔍 Đang xuất file phân tích lỗi: crf_dev_analysis...
⚠️ Cảnh báo: Lệch số lượng từ ở câu 63 (Gốc: 114 | Dự đoán: 106), bỏ qua...
⚠️ Cảnh báo: Lệch số lượng từ ở câu 293 (Gốc: 104 | Dự đoán: 103), bỏ qua...
⚠️ Cảnh báo: Lệch số lượng từ ở câu 1459 (Gốc: 118 | Dự đoán: 75), bỏ qua...
✅ Đã xuất file thành công tại: /content/drive/MyDrive/datasetViMedNER/crf_dev_analysis.txt

🔍 Đang xuất file phân tích lỗi: focal_dev_analysis...
⚠️ Cảnh báo: Lệch số lượng từ ở câu 63 (Gốc: 114 | Dự đoán: 106), bỏ qua...
⚠️ Cảnh báo: Lệch số lượng từ ở câu 293 (Gốc: 104 | Dự đoán: 103), bỏ qua...
⚠️ Cảnh báo: Lệch số lượng từ ở câu 1459 (Gốc: 118 | Dự đoán: 75), bỏ qua...
✅ Đã xuất file thành công tại: /content/drive/MyDrive/datasetViMedNER/focal_dev_analysis.txt
